# vLLM server for the TabLLM pipeline (free Colab T4)

This notebook boots an **OpenAI-compatible [vLLM](https://github.com/vllm-project/vllm) server**
on a free Colab **T4 GPU**, exposes it through a **cloudflared** quick-tunnel, and prints a
public `https://….trycloudflare.com/v1` URL.

On the machine running the analysis pipeline, set that URL as `$VLLM_BASE_URL` and the TabLLM
run becomes **$0** (no Anthropic API billing).

**Before you start:** `Runtime → Change runtime type → Hardware accelerator → T4 GPU`.

> Colab free sessions have a **~12 h hard cap** and disconnect when idle — the keep-alive cell
> below helps, but a sleeping laptop still kills the browser tab. The pipeline's sqlite cache
> makes resumes cheap (completed cells aren't re-queried).

## 1. Install vLLM (CUDA-matched)

Colab preinstalls `torch … +cu128` (CUDA 12.8), but a fresh `pip install vllm` may pull a CUDA-13
build and then crash on import with `ImportError: libcudart.so.13`. Uninstalling both and letting
vLLM pull its own matched CUDA stack fixes it.

**⚠️ After this cell finishes, do `Runtime → Restart session`, then continue from cell 2.**
(The RAPIDS `cudf`/`cuml`/`dask-cuda` dependency-conflict warnings are harmless — those packages
aren't used here.)

In [ ]:
!pip uninstall -y vllm torch torchvision torchaudio
!pip install -q vllm
# then: Runtime -> Restart session

## 2. Launch the vLLM OpenAI server

Started as a background process (via `python -m vllm.entrypoints.openai.api_server` to avoid any
PATH issues), logging to `vllm.log`. We poll `/v1/models` until it answers, and bail early with a
log tail if the process dies during startup.

`MODEL` defaults to a 3B model that runs comfortably on a T4 for full runs. For a stronger
**smoke-only** model, uncomment the AWQ block (see the model menu at the bottom). Whatever you set
here, the pipeline's `--model` / `--tabllm-model` **must match it exactly** — the served id is part
of the response-cache key.

In [ ]:
import sys, subprocess, time, requests

MODEL = "unsloth/Llama-3.2-3B-Instruct"
cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
       "--model", MODEL,
       "--max-model-len", "4096",
       "--gpu-memory-utilization", "0.90",
       "--port", "8000"]

# --- stronger, SMOKE-ONLY option (4-bit AWQ ~9 GB, slow on a T4): ---
# MODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"
# cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
#        "--model", MODEL, "--quantization", "awq", "--dtype", "half",
#        "--max-model-len", "4096", "--gpu-memory-utilization", "0.95", "--port", "8000"]

logf = open("vllm.log", "w")
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
print(f"launching {MODEL} (pid {proc.pid}) — this can take a few minutes on first load ...")

ready = False
for _ in range(180):                       # ~15 min ceiling; loads usually finish well before
    if proc.poll() is not None:            # process died during startup
        print("!! vLLM process EXITED during startup. Last log lines:")
        print("".join(open("vllm.log").readlines()[-25:]))
        break
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=3).ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

if ready:
    print("vLLM READY — /v1/models is answering. Proceed to the tunnel cell.")
elif proc.poll() is None:
    print("!! not ready yet after the wait; check the log tail:")
    print("".join(open("vllm.log").readlines()[-25:]))

## 3. Expose it with a free cloudflared tunnel

Downloads the `cloudflared` binary and opens a quick-tunnel (no signup). Copy the printed
`BASE_URL` and, on the pipeline machine, run `export VLLM_BASE_URL="<that URL>"`.

In [ ]:
import subprocess, time, re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
subprocess.Popen("./cloudflared tunnel --url http://localhost:8000 > tunnel.log 2>&1", shell=True)

url = None
for _ in range(24):                        # ~2 min for the tunnel to come up
    time.sleep(5)
    try:
        m = re.search(r"https://[a-z0-9.-]+\.trycloudflare\.com", open("tunnel.log").read())
    except FileNotFoundError:
        continue
    if m:
        url = m.group(0)
        break

if url:
    print("Tunnel URL :", url)
    print("BASE_URL   :", url + "/v1")
    print()
    print("On the pipeline machine, run:")
    print(f'  export VLLM_BASE_URL="{url}/v1"')
else:
    print("!! tunnel URL not found yet — check tunnel.log:")
    print("".join(open("tunnel.log").readlines()[-15:]))

## 4. (Optional) Keep-alive heartbeat

Colab disconnects idle sessions. Paste this in your **browser's DevTools console** (F12 → Console)
to auto-click the UI periodically. It does **not** survive the laptop going to sleep — that
suspends the whole browser tab.

```javascript
function KeepAlive(){
  const btn = document.querySelector("colab-toolbar-button#connect");
  if (btn) { btn.click(); }
  console.log("keep-alive tick", new Date().toLocaleTimeString());
}
setInterval(KeepAlive, 60000);
```

## Model menu (T4 = 16 GB, Turing / fp16-only)

| Model id (`--model`) | Fits a T4? | Use for |
|---|---|---|
| `unsloth/Llama-3.2-3B-Instruct` | yes (unquantized) | **default** — full learning-curve runs |
| `Qwen/Qwen2.5-7B-Instruct-AWQ` | yes (4-bit AWQ) | stronger classifier, still workable |
| `Qwen/Qwen2.5-14B-Instruct-AWQ` | at a stretch (4-bit AWQ, ~9 GB) | **smoke tests only** — too slow for the full grid |

Rules of thumb on a T4: no bf16 / no fp8 / no FlashAttention-2. Use `--dtype half` and
`--quantization awq` for the AWQ models; drop `--max-model-len` to `2048` if you hit OOM.

**The pipeline-side `--model` / `--tabllm-model` must equal the `MODEL` served here** — it is part
of the sqlite cache key, so a mismatch silently produces a fresh (empty) cache.